# Qwen3-Reranker-0.6B：Colab T4 smoke benchmark

本 notebook 用公开仓库和公开的 `Qwen/Qwen3-Reranker-0.6B` checkpoint 做一个可复现的 5-sample smoke benchmark。运行前在 Colab 选择 `Runtime → Change runtime type → T4 GPU`。

流程：

1. clone 当前公开仓库；
2. 安装与实验环境一致的 pinned 依赖；
3. 通过 Colab upload **只上传** `catalog.jsonl`（`public_set.jsonl` 从仓库读取）；
4. 用 seed 17 生成 target-free、scenario-stratified manifest；
5. 在固定 Hugging Face revision `e61197ed45024b0ed8a2d74b80b4d909f1255473` 下载 checkpoint；
6. 对同一批 5 个 validation samples 跑 feature-only baseline 和 CUDA Top-30 rerank；
7. 下载 manifest、baseline、reranked 和 comparison JSON。

Notebook 不需要 Hugging Face token 或任何 API key。模型下载完成后，benchmark 会强制 offline loading。

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/ByteSize2026/techjam-conversational-search.git"
COLAB_ROOT = Path("/content/techjam")
REPO_DIR = COLAB_ROOT / "repo"
CACHE_ROOT = COLAB_ROOT / "caches"
for cache_path in (
    COLAB_ROOT,
    CACHE_ROOT / "pip",
    CACHE_ROOT / "tmp",
    CACHE_ROOT / "torch",
    COLAB_ROOT / "models" / "huggingface",
):
    cache_path.mkdir(parents=True, exist_ok=True)
os.environ.update({
    "HF_HOME": str(COLAB_ROOT / "models" / "huggingface"),
    "HUGGINGFACE_HUB_CACHE": str(COLAB_ROOT / "models" / "huggingface"),
    "TRANSFORMERS_CACHE": str(COLAB_ROOT / "models" / "huggingface"),
    "PIP_CACHE_DIR": str(CACHE_ROOT / "pip"),
    "TMPDIR": str(CACHE_ROOT / "tmp"),
    "XDG_CACHE_HOME": str(CACHE_ROOT),
    "TORCH_HOME": str(CACHE_ROOT / "torch"),
})

# Re-running the notebook reuses the same clone, but never silently uses a
# different repository. A fresh Colab runtime normally takes the clone path.
if not (REPO_DIR / ".git").is_dir():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    remote = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "remote", "get-url", "origin"],
        text=True,
    ).strip()
    if remote != REPO_URL:
        raise RuntimeError(f"Existing checkout has unexpected origin: {remote}")

os.chdir(REPO_DIR)
assert (REPO_DIR / "data" / "public_set.jsonl").is_file()
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
BENCHMARK_SCRIPT = REPO_DIR / "scripts" / "benchmark_qwen_reranker.py"
if not BENCHMARK_SCRIPT.is_file():
    raise RuntimeError("Public checkout has no Qwen benchmark script; refusing to run.")
benchmark_source = BENCHMARK_SCRIPT.read_text(encoding="utf-8")
required_markers = ("frozen_trace_path", "qwen_reranker_device", "cuda")
missing_markers = [marker for marker in required_markers if marker not in benchmark_source]
help_result = subprocess.run(
    [sys.executable, str(BENCHMARK_SCRIPT), "rerank", "--help"],
    cwd=REPO_DIR,
    text=True,
    capture_output=True,
    check=True,
)
if "--device" not in help_result.stdout or "cuda" not in help_result.stdout:
    missing_markers.append("rerank --device cuda CLI capability")
if missing_markers:
    raise RuntimeError(
        "This public checkout is the legacy pre-CUDA benchmark. Push the CUDA patch to origin/main,"
        " start a fresh Colab runtime, and rerun; missing: " + ", ".join(missing_markers)
    )
print(f"Repository: {REPO_URL}")
print(f"Checkout:   {REPO_DIR}")
print(f"Commit:     {commit}")
print("CUDA benchmark capability check: OK")

## Install pinned runtime

These packages match the validated experiment environment. Colab's pre-installed CUDA-enabled `torch` is deliberately preserved; `--no-cache-dir` keeps pip's cache from consuming additional disk space.

In [ ]:
PINNED_PACKAGES = [
    "transformers==5.15.1",
    "sentence-transformers==6.0.0",
    "huggingface-hub==1.28.0",
    "safetensors==0.8.0",
    "tokenizers==0.22.2",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--no-cache-dir", *PINNED_PACKAGES],
    check=True,
)

pip_check = subprocess.run([sys.executable, "-m", "pip", "check"], text=True, capture_output=True)
if pip_check.returncode:
    print("pip check reported:")
    print(pip_check.stdout or pip_check.stderr)
else:
    print("pip check: OK")
print("Pinned packages:")
for package in PINNED_PACKAGES:
    print("  " + package)

## T4 / CUDA guard

The benchmark is intentionally CUDA-only. A non-T4 NVIDIA GPU is allowed for experimentation, but the reported resource profile is then not the requested T4 profile.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is unavailable. In Colab choose Runtime → Change runtime type → T4 GPU, then restart."
    )
gpu_name = torch.cuda.get_device_name(0)
cuda_version = torch.version.cuda
print(f"GPU: {gpu_name}")
print(f"CUDA runtime: {cuda_version}")
if "t4" not in gpu_name.lower():
    print("WARNING: this is not a Tesla T4; continue only if a different CUDA GPU is intentional.")
else:
    print("T4 guard: OK")
torch.cuda.empty_cache()

## Upload the catalog (only this file)

Do not upload `public_set.jsonl`, model files, credentials, or any other file. The public session file is cloned from the repository; the evaluator is the only component that reads its labels.

In [ ]:
from google.colab import files

uploaded = files.upload()
uploaded_names = sorted(uploaded)
if uploaded_names != ["catalog.jsonl"]:
    raise ValueError(
        f"Upload exactly one file named catalog.jsonl; received {uploaded_names!r}"
    )
catalog_path = REPO_DIR / "data" / "catalog.jsonl"
catalog_bytes = uploaded["catalog.jsonl"]
if not catalog_bytes:
    raise ValueError("catalog.jsonl is empty")
catalog_path.write_bytes(catalog_bytes)
print(f"Wrote {catalog_path} ({catalog_path.stat().st_size:,} bytes)")

## Fixed model snapshot

The revision is an immutable commit hash. `snapshot_download` is called without a token because this is a public checkpoint.

In [ ]:
from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen3-Reranker-0.6B"
MODEL_REVISION = "e61197ed45024b0ed8a2d74b80b4d909f1255473"
MODEL_DIR = COLAB_ROOT / "models" / "qwen3-reranker-0.6b-e61197ed"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = Path(
    snapshot_download(
        repo_id=MODEL_ID,
        revision=MODEL_REVISION,
        local_dir=str(MODEL_DIR),
    )
)
if not (MODEL_PATH / "config.json").is_file():
    raise RuntimeError(f"Checkpoint snapshot is incomplete: {MODEL_PATH}")
print(f"Model:    {MODEL_ID}")
print(f"Revision: {MODEL_REVISION}")
print(f"Path:     {MODEL_PATH}")

# No model/network fetch is allowed after this point.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

## Seed-17 manifest

The manifest is generated before either benchmark run and is shared by baseline and reranking. Only `sample_id`, scenario, and difficulty metadata are copied into it.

In [ ]:
RESULT_DIR = COLAB_ROOT / "results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = RESULT_DIR / "qwen3-reranker-manifest-seed17.json"
subprocess.run(
    [
        sys.executable,
        "scripts/benchmark_qwen_reranker.py",
        "manifest",
        "--public-set",
        str(REPO_DIR / "data" / "public_set.jsonl"),
        "--output",
        str(MANIFEST_PATH),
        "--seed",
        "17",
    ],
    cwd=REPO_DIR,
    check=True,
)
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
if manifest.get("seed") != 17:
    raise RuntimeError(f"Unexpected manifest seed: {manifest.get('seed')!r}")
print(f"Manifest: {MANIFEST_PATH}")
print({name: len(values) for name, values in manifest["split_ids"].items()})

## 5-sample feature-only baseline

This uses the first five IDs of the seed-17 validation split. No model backend is enabled.

In [ ]:
BASELINE_RESULT = RESULT_DIR / "feature-only-baseline-smoke5.json"
subprocess.run(
    [
        sys.executable,
        "scripts/benchmark_qwen_reranker.py",
        "baseline",
        "--catalog",
        str(catalog_path),
        "--public-set",
        str(REPO_DIR / "data" / "public_set.jsonl"),
        "--manifest",
        str(MANIFEST_PATH),
        "--split",
        "validation",
        "--sample-limit",
        "5",
        "--output",
        str(BASELINE_RESULT),
    ],
    cwd=REPO_DIR,
    check=True,
)
baseline = json.loads(BASELINE_RESULT.read_text(encoding="utf-8"))
print(json.dumps({key: baseline.get(key) for key in ("sample_count", "hit_rate_at_10", "mrr", "mttc", "recommended_technical_score")}, indent=2))

## 5-sample Top-30 Qwen rerank

The deterministic agent supplies at most 30 catalog-valid candidates; Qwen only reorders that whitelist. `--device cuda` is required by the guard above. A 60-second soft timeout keeps slow batches from stalling the smoke run; a timeout/failure preserves the feature-only order.

In [ ]:
RERANKED_RESULT = RESULT_DIR / "qwen3-reranked-top30-smoke5.json"
subprocess.run(
    [
        sys.executable,
        "scripts/benchmark_qwen_reranker.py",
        "rerank",
        "--catalog",
        str(catalog_path),
        "--public-set",
        str(REPO_DIR / "data" / "public_set.jsonl"),
        "--manifest",
        str(MANIFEST_PATH),
        "--split",
        "validation",
        "--sample-limit",
        "5",
        "--model-path",
        str(MODEL_PATH),
        "--revision",
        MODEL_REVISION,
        "--device",
        "cuda",
        "--batch-size",
        "8",
        "--candidate-limit",
        "30",
        "--timeout-seconds",
        "60",
        "--fusion-weight",
        "1.0",
        "--output",
        str(RERANKED_RESULT),
    ],
    cwd=REPO_DIR,
    check=True,
)
reranked = json.loads(RERANKED_RESULT.read_text(encoding="utf-8"))
print(json.dumps({key: reranked.get(key) for key in ("sample_count", "hit_rate_at_10", "mrr", "mttc", "recommended_technical_score")}, indent=2))

## Compare and download results

In [ ]:
COMPARISON_RESULT = RESULT_DIR / "qwen3-reranker-comparison-smoke5.json"
subprocess.run(
    [
        sys.executable,
        "scripts/benchmark_qwen_reranker.py",
        "compare",
        "--baseline",
        str(BASELINE_RESULT),
        "--reranked",
        str(RERANKED_RESULT),
        "--manifest",
        str(MANIFEST_PATH),
        "--split",
        "validation",
        "--output",
        str(COMPARISON_RESULT),
    ],
    cwd=REPO_DIR,
    check=True,
)
comparison = json.loads(COMPARISON_RESULT.read_text(encoding="utf-8"))
print(json.dumps(comparison["comparisons"][0]["delta"], indent=2))

RESULT_BUNDLE = COLAB_ROOT / "qwen3-reranker-smoke5-results.zip"
import zipfile
with zipfile.ZipFile(RESULT_BUNDLE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in (MANIFEST_PATH, BASELINE_RESULT, RERANKED_RESULT, COMPARISON_RESULT):
        archive.write(path, arcname=path.name)
print(f"Result bundle: {RESULT_BUNDLE}")
files.download(str(RESULT_BUNDLE))